# 08 - Causal patching, Experiment C with the tuned probe (GPU)

**Settings:** Accelerator **GPU T4 x1** | Internet **On** | Secret `HF_TOKEN` (optional). Code from GitHub (`cf-tfidf`), data from HF (pinned).

**Why:** the layer sweep (notebook 07) showed the deployed probe (LR, C = 1, layer 10) is under-regularised and transfers poorly to the counterfactuals; LR with C = 0.01 at layer 12 is as good on natural failures and much better on induced ones (layer chosen among the CV-tied layers 10-14 by the train-split counterfactuals). Experiments A and B do not use the probe, so only **Experiment C** is re-run, on the **same 219 pairs** as notebook 05.

**Steps**
1. **Train the probe bundle** (CPU part, ~15 min): `src.pipeline.probes` with `--C 0.01`, layer 12, τ and Platt calibration from train-split CV, exactly the deployed recipe otherwise.
2. **Preflight** (~5 min): 2 pairs. Check `HOOK SELF-TEST: PASS` and `PROBE-LINK SELF-CHECK ... -> OK` (cosine ~1.0).
3. **Main run** `--c-only` (baselines, sanity, entity-span patches + probe; ~40 s per pair, ~2.5 h).
4. **Package** results + the new probe bundle -> `patching_c_tuned_results.zip`.

Tip: after the preflight passed, **Save Version -> Save & Run All (Commit)**.

In [ ]:
# ===== Setup: internet, clone, HF token, GPU =====
import os, socket, subprocess, sys, time, zipfile, json
for host in ('github.com', 'huggingface.co'):
    socket.gethostbyname(host)
WORK = '/kaggle/working/LinguaFranca'
if not os.path.exists(WORK):
    subprocess.run(['git', 'clone', '--depth', '1', '-b', 'cf-tfidf', 'https://github.com/ANLPproject/LinguaFranca.git', WORK], check=True)
else:
    subprocess.run(['git', '-C', WORK, 'pull'], check=True)
os.chdir(WORK); sys.path.insert(0, WORK)
print(subprocess.run(['git', 'log', '-1', '--oneline'], capture_output=True, text=True).stdout)
try:
    from kaggle_secrets import UserSecretsClient
    os.environ['HF_TOKEN'] = UserSecretsClient().get_secret('HF_TOKEN')    # read by the scripts from the environment
    print('HF_TOKEN loaded from Kaggle secrets')
except Exception as e:
    print(f'No HF_TOKEN secret ({e.__class__.__name__}); the ungated model mirror will be used')
import torch
assert torch.cuda.is_available(), 'No GPU: Session options -> Accelerator -> GPU T4 x1, then restart.'

def run(cmd):
    '''Stream a command's output; return its exit code.'''
    print('$', ' '.join(cmd), flush=True)
    p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1,
                         env={**os.environ, 'PYTHONUNBUFFERED': '1'})
    for line in p.stdout:
        if 'it/s]' in line or 's/it]' in line:
            continue
        print(line, end='')
    return p.wait()

PROBE = '/kaggle/working/probes_c001_L12'
OUT = '/kaggle/working/patching_tfidf_c_tuned'
PRE = '/kaggle/working/patching_c_preflight'

In [ ]:
# ===== STEP 1 - train the tuned probe bundle (C = 0.01, layer 12; deployed recipe otherwise) =====
if not os.path.exists(f'{PROBE}/probes_meta.json'):
    rc = run([sys.executable, '-m', 'src.pipeline.probes', '--setting', 'tfidf', '--out', PROBE, '--C', '0.01',
              '--hop1-layer', '12', '--hop2-layer', '12', '--hop12-layer', '12', '--sweep', 'none'])
    assert rc == 0, 'probe training failed - see above'
m = json.load(open(f'{PROBE}/probes_meta.json'))
p = m['probes']['hop12']
print('C', m['C'], '| hop12 layer', p['layer'], '| tau', round(p['tau'], 3), '| test AUROC', round(p['test']['auroc'], 3),
      '| hedge recall', p['test_strata']['hedge_fail_recall'], '| confident recall', round(p['test_strata']['confident_fail_recall'], 3))
assert m['C'] == 0.01 and p['layer'] == 12

In [ ]:
# ===== STEP 2 - PREFLIGHT (2 pairs, 2 blocks): hook self-test + probe-link self-check =====
rc = run([sys.executable, 'causal_patching.py', '--c-only', '--probe-dir', PROBE, '--out', PRE, '--max-pairs', '2',
          '--layers', '0,12', '--max-new-tokens', '40'])
assert rc == 0 and os.path.exists(f'{PRE}/summary.md'), 'PREFLIGHT FAILED - read the log above.'
print('\nPREFLIGHT PASSED')

In [ ]:
# ===== STEP 3 - MAIN RUN: Experiment C on the same 219 pairs as notebook 05 (resumable) =====
BUDGET_MIN = 600
t_start = time.time()
for attempt in range(1, 4):
    left = BUDGET_MIN - (time.time() - t_start) / 60
    if left < 10:
        break
    rc = run([sys.executable, 'causal_patching.py', '--c-only', '--probe-dir', PROBE, '--out', OUT,
              '--max-pairs', '219', '--max-minutes', f'{left:.0f}'])
    if rc == 0:
        break
    print(f'\n[attempt {attempt}] exit code {rc} - resuming')
print('main run exit code:', rc)

In [ ]:
# ===== STEP 4 - package (always run this): results + the probe bundle =====
run([sys.executable, 'causal_patching.py', '--out', OUT, '--summarize-only'])
zpath = '/kaggle/working/patching_c_tuned_results.zip'
with zipfile.ZipFile(zpath, 'w', zipfile.ZIP_DEFLATED) as z:
    for d in (OUT, PRE, PROBE):
        for root, _, files in os.walk(d):
            for f in files:
                if f.endswith('.joblib'):
                    continue
                p = os.path.join(root, f)
                z.write(p, os.path.relpath(p, os.path.dirname(d)))
print(f'{zpath}: {os.path.getsize(zpath) / 1e6:.1f} MB')